In [1]:
# # Cell 1: unzip into a writable folder
# !ls /kaggle/input
# !mkdir -p /kaggle/working/proj
# # !unzip -q /kaggle/input/eeg-nipi-code/eeg_nipi_v2.zip -d /kaggle/working/proj
# !find /kaggle/working/proj -name train.py

In [2]:
# import os
# import shutil

# input_dir = '/kaggle/input'
# datasets = os.listdir(input_dir)
# print(f"Datasets found in Kaggle input: {datasets}")

# if datasets:
#     # Get the exact path of the first dataset attached
#     dataset_path = os.path.join(input_dir, datasets[0]) 
#     dest_path = '/kaggle/working/proj'
    
#     # Clean up destination if it exists from previous failed attempts
#     if os.path.exists(dest_path):
#         shutil.rmtree(dest_path)
        
#     print(f"Copying files from {dataset_path} to {dest_path}...")
#     shutil.copytree(dataset_path, dest_path)
    
#     # Search for train.py to confirm success
#     for root, dirs, files in os.walk(dest_path):
#         if 'train.py' in files:
#             print(f"Success! train.py is located in: {root}")
# else:
#     print("Error: No datasets found. Check the right sidebar to ensure the dataset is attached.")
    

In [3]:
# import os

# # Automatically locate train.py and change to its directory
# train_script = !find /kaggle/working/proj -name train.py

# if train_script:
#     target_dir = os.path.dirname(train_script[0])
#     print(f"Navigating to: {target_dir}")
#     %cd {target_dir}
# else:
#     print("Error: train.py was not found.")

# !pip -q install PyWavelets mne
# !python -c "import torch;print(torch.__version__, torch.cuda.is_available())"

In [4]:
# import os

# # 1. Automatically find and navigate to the directory containing train.py
# train_script = !find /kaggle/working/proj -name train.py
# if train_script:
#     target_dir = os.path.dirname(train_script[0])
#     print(f"Working directory set to: {target_dir}")
#     %cd {target_dir}
# else:
#     print("Error: train.py not found.")

# # 2. Run the specific NIPI model training configurations
# !python train.py --synthetic --epochs 80 --n 1500 --no-line --lam-cov 2.0 --seed 1 --tag _v2s1
# !python train.py --synthetic --epochs 80 --n 1500 --no-line --lam-cov 2.0 --seed 2 --tag _v2s2

# # 3. Zip the outputs for download
# !zip -qr /kaggle/working/results.zip outputs
# print("Training complete and results packed into /kaggle/working/results.zip")

In [5]:
# import os

# # Find where data_real.py is and navigate there
# script_path = !find /kaggle/working/proj -name data_real.py
# if script_path:
#     target_dir = os.path.dirname(script_path[0])
#     print(f"Navigating to: {target_dir}")
#     %cd {target_dir}
# else:
#     print("Error: data_real.py not found.")

# # Create the data folder so the scripts have a place to save the .npz files
# !mkdir -p data

In [6]:
# import glob, os

# # 1. Write the bulletproof version of data_real.py
# new_code = """import argparse, glob, numpy as np, mne
# from eegnipi.montage import CH

# ALIAS = {'T7': 'T3', 'T8': 'T4', 'P7': 'T5', 'P8': 'T6'}

# def norm(n):
#     # Strip 'EEG', periods, and Siena '-REF'/'-LE' suffixes, then uppercase
#     n = n.upper().replace('EEG', '').replace('.', '').replace('-REF', '').replace('-LE', '').strip()
#     return ALIAS.get(n, n)

# def load(edf, mode, fs=128, dur=4.0, stim=1.0, max_min=None):
#     raw = mne.io.read_raw_edf(edf, preload=False, verbose='error')
#     raw.rename_channels(lambda x: x.replace('.', '')) # Kaggle PhysioNet patch
    
#     # Safely match normalized dataset channels against normalized target channels
#     names = {c: norm(c) for c in raw.ch_names}
#     pick = []
#     for w in CH:
#         target = norm(w)
#         try:
#             match = next(k for k, v in names.items() if v == target)
#             pick.append(match)
#         except StopIteration:
#             raise ValueError(f"Missing channel: '{w}'. Dataset has: {list(names.values())}")
            
#     raw.pick(pick); raw.reorder_channels(pick)
#     if max_min: raw.crop(0, min(raw.times[-1], max_min * 60))
#     raw.load_data(verbose='error'); raw.filter(1, 45, verbose='error'); raw.resample(fs, verbose='error')
#     T = int(fs * dur); X, y = [], []
    
#     if mode == 'mi':
#         for o, d in zip(raw.annotations.onset, raw.annotations.description):
#             if d in ('T1', 'T2'):
#                 s = int(round((o - stim) * fs))
#                 if s >= 0 and s + T <= raw.n_times: 
#                     X.append(raw.get_data(start=s, stop=s + T) * 1e6); y.append(int(d == 'T2'))
#     else:
#         D = raw.get_data() * 1e6
#         for s in range(0, D.shape[1] - T + 1, T): X.append(D[:, s:s + T]); y.append(0)
#     return np.array(X, np.float32).reshape(-1, len(CH), T), np.array(y, int)

# def is_clean(e, fs=128):
#     from scipy.signal import welch
#     f, P = welch(e, fs, nperseg=128, axis=-1); hf = P[:, f > 25].sum(1) / P.sum(1)
#     return (np.ptp(e, axis=-1).max() < 100) and (hf.max() < 0.15)

# if __name__ == '__main__':
#     p = argparse.ArgumentParser(); p.add_argument('--edf', nargs='+'); p.add_argument('--out'); p.add_argument('--mode', default='mi')
#     p.add_argument('--max-min', type=float, default=None); a = p.parse_args()
#     files = sorted({f for pat in a.edf for f in glob.glob(pat)}); print(len(files), 'files found')
#     XS, YS = [], []
#     for f in files:
#         try: 
#             x, y = load(f, a.mode, max_min=a.max_min); XS.append(x); YS.append(y)
#             print(' ', f.split('/')[-1], len(x), 'epochs')
#         except Exception as e: print('  SKIPPED', f.split('/')[-1], '->', str(e)[:150])
    
#     if len(XS) == 0:
#         print("ERROR: All files were skipped.")
#         exit(1)
        
#     X, y = np.concatenate(XS), np.concatenate(YS); keep = np.array([is_clean(e) for e in X]); X, y = X[keep], y[keep]
#     print(f'kept {keep.sum()}/{len(keep)} artifact-free epochs; class counts: {np.bincount(y)}; shape {X.shape}; std {X.std():.1f} uV')
#     import os; os.makedirs(os.path.dirname(a.out) or '.', exist_ok=True); np.savez(a.out, X=X, y=y, fs=128)
# """

# with open('data_real.py', 'w') as f:
#     f.write(new_code)
# print("Saved corrected data_real.py\n")

# # 2. Extract PhysioNet (Motor Imagery)
# print("--- Processing PhysioNet ---")
# files = []
# for f in glob.glob('/kaggle/input/**/S*R*.edf', recursive=True):
#     b = os.path.basename(f)
#     if b[4:7] in ('R04', 'R08', 'R12') and 1 <= int(b[1:4]) <= 20: 
#         files.append(f)

# if files:
#     !python data_real.py --mode mi --out data/mi_clean.npz --edf {' '.join(sorted(files))}
# else:
#     print("0 PhysioNet files found.")

# # 3. Extract Siena Dataset
# print("\n--- Processing Siena ---")
# siena = sorted(glob.glob('/kaggle/input/**/PN0*-*.edf', recursive=True))[:6]

# if siena:
#     !python data_real.py --mode continuous --max-min 10 --out data/siena_clean.npz --edf {' '.join(siena)}
# else:
#     print("0 Siena files found.")

In [7]:
# import numpy as np, glob, sys
# sys.path.insert(0, '.')
# from data_real import load
# from scipy.signal import welch
# def stats(X, fs=128):
#     ptp = np.ptp(X, axis=-1).max(1)                              # worst channel peak-to-peak per epoch
#     f, P = welch(X, fs, nperseg=128, axis=-1); hf = (P[..., f > 25].sum(-1) / P.sum(-1)).max(1)
#     return ptp, hf
# sets = {'MI': [load(f, 'mi')[0] for f in sorted(glob.glob('/kaggle/input/**/S0[0-1][0-9]R04.edf', recursive=True))[:6]],
#         'Siena': [load(f, 'continuous', max_min=10)[0] for f in sorted(glob.glob('/kaggle/input/**/PN0*-*.edf', recursive=True))[:3]]}
# for name, L in sets.items():
#     X = np.concatenate(L); ptp, hf = stats(X); print(name, 'epochs', len(X))
#     print('  peak-to-peak uV, percentiles 10/25/50/75/90:', np.percentile(ptp, [10, 25, 50, 75, 90]).round(0))
#     print('  HF fraction, percentiles 10/25/50/75/90:', np.percentile(hf, [10, 25, 50, 75, 90]).round(3))
#     for p_, h_ in [(100, .15), (150, .15), (100, .25), (150, .25), (200, .30)]:
#         print(f'  keep if ptp<{p_} and hf<{h_}:', ((ptp < p_) & (hf < h_)).mean().round(2))

In [8]:
# import numpy as np, glob, os, sys
# sys.path.insert(0, '.')
# from data_real import load
# from scipy.signal import welch
# os.makedirs('data', exist_ok=True)
# def collect(files, mode, **kw):
#     Xs, Ys, S = [], [], []
#     for f in files:
#         try: x, y = load(f, mode, **kw); Xs.append(x); Ys.append(y); S += [os.path.basename(f)[:4]] * len(x)
#         except Exception as e: print('skip', os.path.basename(f), str(e)[:80])
#     return np.concatenate(Xs), np.concatenate(Ys), np.array(S)
# mi = sorted(f for f in glob.glob('/kaggle/input/**/S*R*.edf', recursive=True)
#             if os.path.basename(f)[4:7] in ('R04', 'R08', 'R12') and 1 <= int(os.path.basename(f)[1:4]) <= 20)
# Xm, ym, sm = collect(mi, 'mi'); np.savez('data/mi_all.npz', X=Xm, y=ym, subj=sm, fs=128)
# sie = sorted(glob.glob('/kaggle/input/**/PN0*-*.edf', recursive=True))[:6]
# Xs, ys, ss = collect(sie, 'continuous', max_min=10); np.savez('data/siena_all.npz', X=Xs, y=ys, subj=ss, fs=128)
# def stats(X, q):
#     ptp = np.quantile(np.ptp(X, axis=-1), q, axis=1)
#     f, P = welch(X, 128, nperseg=128, axis=-1); hf = np.quantile(P[..., f > 25].sum(-1) / P.sum(-1), q, axis=1)
#     return ptp, hf
# for name, X in [('MI', Xm), ('Siena', Xs)]:
#     print(name, len(X), 'epochs')
#     for q in (1.0, 0.9, 0.75):
#         ptp, hf = stats(X, q)
#         print(f'  channel quantile {q}: ptp pct10/50/90', np.percentile(ptp, [10, 50, 90]).round(0), ' hf pct10/50/90', np.percentile(hf, [10, 50, 90]).round(2))
#         for p_, h_ in [(100, .15), (150, .25), (200, .30), (300, .35)]:
#             print(f'    keep if ptp<{p_} and hf<{h_}: {((ptp < p_) & (hf < h_)).mean():.2f}')

In [9]:
# import numpy as np
# from scipy.signal import welch
# def keep_mask(X, q=0.9, ptp_max=150, hf_max=0.25, cap=500):
#     p = np.ptp(X, axis=-1)
#     f, P = welch(X, 128, nperseg=128, axis=-1); hf = P[..., f > 25].sum(-1) / P.sum(-1)
#     return (np.quantile(p, q, axis=1) < ptp_max) & (np.quantile(hf, q, axis=1) < hf_max) & (p.max(1) < cap)
# for name in ('mi', 'siena'):
#     d = np.load(f'data/{name}_all.npz'); m = keep_mask(d['X'])
#     print(name, 'kept', m.sum(), 'of', len(m), '| classes', np.bincount(d['y'][m]), '| subjects', len(set(d['subj'][m])), '| std uV', d['X'][m].std().round(1))
#     np.savez(f'data/{name}_clean.npz', X=d['X'][m], y=d['y'][m], fs=128)

In [10]:
# import os, shutil
# dest_path = '/kaggle/working/proj'

# # Clear old code and copy the newly updated dataset
# if os.path.exists(dest_path): shutil.rmtree(dest_path)
# !mkdir -p {dest_path}
# !cp -r /kaggle/input/eeg*/* {dest_path}/

# # Navigate to the correct directory
# train_script = !find /kaggle/working/proj -name train.py
# if train_script:
#     %cd {os.path.dirname(train_script[0])}
#     print("Successfully loaded updated train.py!")
# else:
#     print("Error: train.py not found.")

In [11]:
# import numpy as np, glob, os, sys, collections
# sys.path.insert(0, '.')
# from data_real import load
# from scipy.signal import welch
# def keep_mask(X, q=0.9, ptp_max=150, hf_max=0.25, cap=500):
#     p = np.ptp(X, axis=-1)
#     f, P = welch(X, 128, nperseg=128, axis=-1); hf = P[..., f > 25].sum(-1) / P.sum(-1)
#     return (np.quantile(p, q, axis=1) < ptp_max) & (np.quantile(hf, q, axis=1) < hf_max) & (p.max(1) < cap)
# by = collections.defaultdict(list)
# for f in sorted(glob.glob('/kaggle/input/**/PN*-*.edf', recursive=True)): by[os.path.basename(f)[:4]].append(f)
# print({k: len(v) for k, v in by.items()})
# sie = [f for k in sorted(by) for f in by[k][:2]]
# Xs, S = [], []
# for f in sie:
#     try: x, _ = load(f, 'continuous', max_min=10); Xs.append(x); S += [os.path.basename(f)[:4]] * len(x)
#     except Exception as e: print('skip', os.path.basename(f), str(e)[:80])
# X = np.concatenate(Xs); S = np.array(S); m = keep_mask(X)
# print('siena kept', m.sum(), 'of', len(m), '| patients', len(set(S[m])), '| per patient', dict(collections.Counter(S[m])), '| std uV', X[m].std().round(1))

# # Create the 'data' directory if it doesn't exist
# os.makedirs('data', exist_ok=True) 
# np.savez('data/siena_clean.npz', X=X[m], y=np.zeros(m.sum(), int), subj=S[m], fs=128)

In [12]:
# import os, shutil

# input_dir = None
# for root, dirs, files in os.walk('/kaggle/input'):
#     if 'train.py' in files:
#         input_dir = root
#         break

# dest_path = '/kaggle/working/proj'
# os.makedirs(dest_path, exist_ok=True)

# if input_dir:
#     shutil.copytree(input_dir, dest_path, dirs_exist_ok=True)
#     os.chdir(dest_path)
#     print(f"Code copied successfully. Working directory is now: {os.getcwd()}")
    
#     if os.path.exists('data/siena_clean.npz') and os.path.exists('data/mi_clean.npz'):
#         print("Clean data arrays found! Proceed to Step 2.")
#     else:
#         print("WARNING: Clean data missing. You may need to re-run the data extraction.")
# else:
#     print("Error: train.py not found in /kaggle/input/")

In [13]:
# nipi_path = 'eegnipi/nipi.py'

# if os.path.exists(nipi_path):
#     with open(nipi_path, 'r') as f:
#         lines = f.readlines()
    
#     with open(nipi_path, 'w') as f:
#         for line in lines:
#             f.write(line)
#             if line.strip().startswith('def task_retention('):
#                 f.write("    import numpy as np\n    if len(np.unique(labels)) < 2: return np.nan, np.nan, np.nan\n")
                
#     print("Patched eegnipi/nipi.py successfully. The single-class crash is fixed. Proceed to Step 3.")
# else:
#     print(f"Error: {nipi_path} not found.")

In [14]:
# print("Starting cross-dataset training...")
# !python train.py --data data/siena_clean.npz --test-data data/mi_clean.npz --epochs 80 --no-line --lam-cov 2.0 --tag _real

In [15]:
# import pandas as pd
# d = pd.read_csv('outputs/final_comparison_real.csv')
# for v in ('ARR', 'NIPI', 'Joint'):
#     print(v); print(d.pivot_table(index='method', columns='kind', values=v).round(3))
# nl = d[d.kind != 'line']
# print('excluding line noise'); print(nl.groupby('method')[['ARR', 'CC', 'NIPI', 'Joint']].mean().round(3).sort_values('Joint', ascending=False))
# print(d[['method', 'task', 'erp']].groupby('method').mean().round(3))

In [16]:
# import numpy as np, glob, os, sys
# sys.path.insert(0, '.')
# from data_real import load
# from scipy.signal import welch
# def keep_mask(X, q=0.9, ptp_max=150, hf_max=0.25, cap=500):
#     p = np.ptp(X, axis=-1)
#     f, P = welch(X, 128, nperseg=128, axis=-1); hf = P[..., f > 25].sum(-1) / P.sum(-1)
#     return (np.quantile(p, q, axis=1) < ptp_max) & (np.quantile(hf, q, axis=1) < hf_max) & (p.max(1) < cap)
# mi2 = sorted(f for f in glob.glob('/kaggle/input/**/S*R*.edf', recursive=True)
#              if os.path.basename(f)[4:7] in ('R04', 'R08', 'R12') and 21 <= int(os.path.basename(f)[1:4]) <= 60)
# Xs, S = [], []
# for f in mi2:
#     try: x, _ = load(f, 'mi'); Xs.append(x); S += [os.path.basename(f)[:4]] * len(x)
#     except Exception as e: print('skip', os.path.basename(f), str(e)[:80])
# Xm2 = np.concatenate(Xs); S = np.array(S); m = keep_mask(Xm2)
# print('MI training subjects: kept', m.sum(), 'of', len(m), '| subjects', len(set(S[m])))
# sie = np.load('data/siena_clean.npz')
# X = np.concatenate([sie['X'], Xm2[m]]); y = np.zeros(len(X), int)
# np.savez('data/train_mix.npz', X=X, y=y, fs=128); print('train_mix', X.shape)

In [17]:
# import numpy as np
# from scipy.signal import welch
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import StratifiedKFold, cross_val_score
# from sklearn.pipeline import make_pipeline
# from sklearn.preprocessing import StandardScaler
# from eegnipi.montage import IDX
# d = np.load('data/mi_clean.npz'); X, y = d['X'], d['y']
# f, P = welch(X, 128, nperseg=128, axis=-1)
# def bp(lo, hi): return np.log(P[..., (f >= lo) & (f < hi)].mean(-1) + 1e-12)
# mu, be = bp(8, 13), bp(13, 30); c3, c4, cz = IDX['C3'], IDX['C4'], IDX['Cz']
# feats = {'lateral mu+beta': np.c_[mu[:, c4] - mu[:, c3], be[:, c4] - be[:, c3]],
#          'C3/C4/Cz mu+beta': np.c_[mu[:, [c3, c4, cz]], be[:, [c3, c4, cz]]],
#          'all-channel mu+beta': np.c_[mu, be]}
# cv = StratifiedKFold(5, shuffle=True, random_state=0)
# for k, F in feats.items():
#     a = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000)), F, y, cv=cv).mean()
#     print(f'{k}: accuracy {a:.2f}  (chance 0.50)')

In [18]:
# !python train.py --data data/train_mix.npz --test-data data/mi_clean.npz --epochs 80 --no-line --lam-cov 2.0 --tag _real2

In [19]:
# import traceback, importlib, sys
# s = open('data_real.py').read()
# print('channel-matching line:', [l.strip()[:110] for l in s.splitlines() if 'pick' in l and 'next(' in l or 'target = norm' in l])
# if 'v == w' in s:
#     s = s.replace('v == w', 'v == norm(w)'); open('data_real.py', 'w').write(s); print('PATCHED the old channel matching')
# for k in [k for k in sys.modules if k == 'data_real']: del sys.modules[k]
# import data_real
# try: x, _ = data_real.load(files[0], 'continuous'); print('OK', x.shape)
# except Exception: traceback.print_exc()

In [20]:
# import numpy as np, glob, os
# from data_real import load
# from eegnipi.nipi import _features
# from scipy.signal import welch
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import StratifiedKFold, cross_val_score
# from sklearn.pipeline import make_pipeline
# from sklearn.preprocessing import StandardScaler
# def keep_mask(X, q=0.9, ptp_max=150, hf_max=0.25, cap=500):
#     p = np.ptp(X, axis=-1)
#     f, P = welch(X, 128, nperseg=128, axis=-1); hf = P[..., f > 25].sum(-1) / P.sum(-1)
#     return (np.quantile(p, q, axis=1) < ptp_max) & (np.quantile(hf, q, axis=1) < hf_max) & (p.max(1) < cap)
# sid = lambda f: int(os.path.basename(f)[1:4])
# files = sorted(f for f in glob.glob('/kaggle/input/**/S*R0[12].edf', recursive=True) if sid(f) <= 20 or sid(f) >= 61)
# Xs, ys = [], []
# for f in files:
#     try: x, _ = load(f, 'continuous'); Xs.append(x); ys += [0 if 'R01' in os.path.basename(f) else 1] * len(x)
#     except Exception as e: print('skip', os.path.basename(f), repr(e)[:150])
# X = np.concatenate(Xs); y = np.array(ys); m = keep_mask(X); print('kept', m.sum(), 'of', len(m), '| per class', np.bincount(y[m])); X, y = X[m], y[m]
# n = min(np.bincount(y)); rng = np.random.default_rng(0)
# idx = np.sort(np.concatenate([rng.choice(np.where(y == c)[0], n, replace=False) for c in (0, 1)])); X, y = X[idx], y[idx]
# acc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000)), _features(X, 128), y, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()
# print('eyes-open/closed epochs', len(y), '| classes', np.bincount(y), '| clean decoding accuracy', round(acc, 2))
# np.savez('data/eoec_clean.npz', X=X, y=y, fs=128)

In [21]:
# def rep(path, a, b):
#     s = open(path).read()
#     if b in s: print('already patched:', a[:40]); return
#     assert a in s, ('NOT FOUND', a[:60]); open(path, 'w').write(s.replace(a, b, 1)); print('patched:', a[:40])
# rep('train.py', "p.add_argument('--test-data', default=None);", "p.add_argument('--test-data', default=None); p.add_argument('--drop', nargs='*', default=[]); p.add_argument('--eval-tag', default=None);")
# rep('train.py', "from eegnipi.nipi import nipi, artifact_suppression, joint_score", "from eegnipi.nipi import nipi, artifact_suppression, joint_score, COMPONENTS")
# rep('train.py', "print('train', len(tr), 'val', len(va), 'test', len(Xte))", "print('train', len(tr), 'val', len(va), 'test', len(Xte)); PARTS = [c for c in COMPONENTS if c not in a.drop]")
# rep('train.py', "evaluate_methods(Xte, yte, methods, fs, kind, snr, verbose=False)", "evaluate_methods(Xte, yte, methods, fs, kind, snr, verbose=False, parts=PARTS)")
# rep('train.py', "res.to_csv(f'outputs/final_comparison{a.tag}.csv', index=False)", "res.to_csv(f'outputs/final_comparison{a.eval_tag if a.eval_tag is not None else a.tag}.csv', index=False)")
# rep('eegnipi/evaluate.py', "seed=1, verbose=True):", "seed=1, verbose=True, parts=None):")
# rep('eegnipi/evaluate.py', "n = nipi(clean, den, labels, fs)", "n = nipi(clean, den, labels, fs, parts=parts) if parts else nipi(clean, den, labels, fs)")
# rep('eegnipi/evaluate.py', "{k: n[k] for k in", "{k: n.get(k, float('nan')) for k in")

In [22]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 0 --no-line --tag _real2 --eval-tag _real2_eoec --drop erp

In [23]:
# import pandas as pd
# d = pd.read_csv('outputs/final_comparison_real2_eoec.csv')
# for v in ('NIPI', 'Joint'):
#     print(v); print(d.pivot_table(index='method', columns='kind', values=v).round(3))
# nl = d[d.kind != 'line']
# print('excluding line noise'); print(nl.groupby('method')[['ARR', 'CC', 'NIPI', 'Joint']].mean().round(3).sort_values('Joint', ascending=False))
# print('components, excluding line'); print(nl.groupby('method')[['spectral', 'phase', 'connect', 'geometry', 'entropy', 'task']].mean().round(3))

In [24]:
# import numpy as np, torch, matplotlib.pyplot as plt, sys
# sys.path.insert(0, '.')
# from eegnipi.model import NIPDNet
# import eegnipi.artifacts as AM
# from eegnipi.baselines import ica, nlms
# from eegnipi.montage import IDX
# from scipy.signal import welch
# AM.MIXED = ['eog', 'emg']; fs = 128
# X = np.load('data/mi_clean.npz')['X'][:60]
# noisy, art, refs = AM.contaminate(X, 'mixed', 0, fs, seed=5)
# net = NIPDNet(); net.load_state_dict(torch.load('outputs/best_nipdnet_real2.pt', map_location='cpu')); net.eval()
# with torch.no_grad(): prop = net(torch.tensor(noisy))[0].numpy()
# outs = {'Clean (reference)': X, 'Noisy': noisy, 'ICA': ica(noisy, refs, fs), 'NLMS': nlms(noisy, refs, fs), 'Proposed': prop}
# i = 3; t = np.arange(X.shape[-1]) / fs
# fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
# for a_, ch in zip(ax, ['Fp1', 'C3', 'Pz']):
#     for k, v in outs.items(): a_.plot(t, v[i, IDX[ch]], label=k, lw=1.6 if k == 'Proposed' else 0.9)
#     a_.set_ylabel(ch + ' (uV)')
# ax[0].legend(ncol=5, fontsize=8); ax[-1].set_xlabel('time (s)'); plt.tight_layout(); plt.savefig('outputs/real_epoch_example.png', dpi=150)
# fig, a_ = plt.subplots(figsize=(6, 4))
# for k, v in outs.items():
#     f, P = welch(v, fs, nperseg=128, axis=-1); a_.semilogy(f, P.mean((0, 1)), label=k)
# a_.set_xlim(0, 45); a_.set_xlabel('Hz'); a_.set_ylabel('PSD'); a_.legend(fontsize=8); plt.tight_layout(); plt.savefig('outputs/real_psd.png', dpi=150)
# print('saved outputs/real_epoch_example.png and outputs/real_psd.png')

In [25]:
# sty = {'Clean (reference)': dict(color='k', lw=2.6, alpha=0.30), 'Noisy': dict(color='tab:orange', lw=0.9),
#        'ICA': dict(color='tab:green', lw=0.9, ls='--'), 'NLMS': dict(color='tab:red', lw=0.9, ls=':'), 'Proposed': dict(color='tab:purple', lw=1.7)}
# fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
# for a_, ch in zip(ax, ['Fp1', 'C3', 'Pz']):
#     for k, v in outs.items(): a_.plot(t, v[i, IDX[ch]], label=k, **sty[k])
#     a_.set_ylabel(ch + ' (uV)')
# ax[0].legend(ncol=5, fontsize=8); ax[-1].set_xlabel('time (s)'); plt.tight_layout(); plt.savefig('outputs/real_epoch_example2.png', dpi=150)
# fig, a_ = plt.subplots(figsize=(6, 4))
# for k, v in outs.items():
#     f, P = welch(v, fs, nperseg=128, axis=-1); a_.semilogy(f, P.mean((0, 1)), label=k, **sty[k])
# a_.set_xlim(1, 45); a_.set_ylim(0.5, 200); a_.set_xlabel('Hz'); a_.set_ylabel('PSD'); a_.legend(fontsize=8); plt.tight_layout(); plt.savefig('outputs/real_psd2.png', dpi=150)

In [26]:
# def rep(path, a, b):
#     s = open(path).read()
#     if b in s: print('already patched'); return
#     assert a in s, ('NOT FOUND', a[:60]); open(path, 'w').write(s.replace(a, b, 1)); print('patched')
# rep('train.py', "p.add_argument('--drop', nargs='*', default=[]);", "p.add_argument('--drop', nargs='*', default=[]); p.add_argument('--zero', nargs='*', default=[]);")
# rep('train.py', "lam = dict(rec=1.0,", "lam = dict(rec=1.0,")
# s = open('train.py').read()
# if "for k_ in a.zero" not in s:
#     s = s.replace("MAP = dict(", "for k_ in a.zero: lam[k_] = 0.0\nMAP = dict(", 1)
#     s = s.replace("if comp in r and not np.isnan(r[comp]): lam[k]", "if comp in r and not np.isnan(r[comp]) and k not in a.zero: lam[k]", 1)
#     open('train.py', 'w').write(s)
# print('zero option present:', "for k_ in a.zero" in open('train.py').read(), "k not in a.zero" in open('train.py').read())

In [27]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --lam-cov 2.0 --drop erp --zero info --tag _abl_info

In [28]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --lam-cov 2.0 --drop erp --zero phase conn --tag _abl_phaseconn

In [29]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --lam-cov 2.0 --drop erp --zero cov --tag _abl_cov

In [30]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --lam-cov 2.0 --drop erp --zero spec --tag _abl_spec

In [31]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --lam-cov 2.0 --drop erp --zero graph --tag _abl_graph

In [32]:
# def rep(path, a, b):
#     s = open(path).read()
#     if b in s: print('already patched'); return
#     assert a in s, ('NOT FOUND', a[:60]); open(path, 'w').write(s.replace(a, b, 1)); print('patched')
# rep('train.py', "p.add_argument('--zero', nargs='*', default=[]);", "p.add_argument('--zero', nargs='*', default=[]); p.add_argument('--lam-scale', type=float, default=1.0);")
# rep('train.py', "for k_ in a.zero: lam[k_] = 0.0", "for k_ in lam:\n    if k_ not in ('rec', 'type'): lam[k_] *= a.lam_scale\nfor k_ in a.zero: lam[k_] = 0.0")
# rep('train.py', "0.05, 5.0)", "0.05, 5.0 * max(1.0, a.lam_scale))")

In [33]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --drop erp --zero spec phase conn cov graph info --tag _abl_none

In [34]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --lam-cov 2.0 --lam-scale 4 --drop erp --tag _lamx4

In [35]:
# !python train.py --data data/train_mix.npz --test-data data/mi_clean.npz --epochs 0 --no-line --tag _lamx4 --eval-tag _lamx4_mi

In [36]:
# !python train.py --data data/train_mix.npz --test-data data/mi_clean.npz --epochs 0 --no-line --tag _abl_none --eval-tag _abl_none_mi

In [37]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --lam-cov 2.0 --lam-scale 4 --drop erp --seed 1 --tag _lamx4_s1

In [38]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --lam-cov 2.0 --lam-scale 4 --drop erp --seed 2 --tag _lamx4_s2

In [39]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --drop erp --zero spec phase conn cov graph info --seed 1 --tag _abl_none_s1

In [40]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --drop erp --zero spec phase conn cov graph info --seed 2 --tag _abl_none_s2

In [41]:
# %%writefile physio_validation.py
# """Stage-3 PHYSIOLOGICAL VALIDATION on NATURALLY contaminated real epochs (no clean reference needed).
# Epochs that failed the clean-epoch rule are 'contaminated'. For every method we ask: are known neural biomarkers preserved
# while real artifacts are suppressed?
#   EO/EC (PhysioNet R01/R02):  (a) alpha reactivity (EC - EO, occipital) vs the value measured on clean epochs,
#                               (b) EO-vs-EC decoding accuracy, subject-wise GroupKFold, on ALL epochs and on CONTAMINATED epochs only,
#                               (c) on contaminated epochs, ratio to raw: frontal 1-4 Hz (blink), temporal 30-45 Hz (muscle),
#                                   occipital alpha (should stay ~1), central mu (should stay ~1)
#   Motor imagery:               left-vs-right mu lateralisation effect size (Cohen d) on all / contaminated epochs
# usage:
#   python physio_validation.py --eoec data/eoec_all.npz --mi data/mi_all.npz --ckpt outputs/best_nipdnet_lamx4.pt --out outputs/physio
#   (omit --ckpt to run the classical methods only; omit --eoec or --mi to skip that part)"""
# import argparse, os, numpy as np, pandas as pd
# from scipy.signal import welch
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import GroupKFold, cross_val_score
# from sklearn.pipeline import make_pipeline
# from sklearn.preprocessing import StandardScaler
# from eegnipi.montage import IDX
# from eegnipi.nipi import _features
# import eegnipi.baselines as B
# FS = 128

# def keep_mask(X, q=0.9, ptp_max=150, hf_max=0.25, cap=500):        # the SAME clean-epoch rule used everywhere else
#     p = np.ptp(X, axis=-1)
#     f, P = welch(X, FS, nperseg=128, axis=-1); hf = P[..., f > 25].sum(-1) / P.sum(-1)
#     return (np.quantile(p, q, axis=1) < ptp_max) & (np.quantile(hf, q, axis=1) < hf_max) & (p.max(1) < cap)

# def bandpow(X, lo, hi):
#     f, P = welch(X, FS, nperseg=128, axis=-1); return P[..., (f >= lo) & (f < hi)].mean(-1)       # (N, C)
# def chs(names): return [IDX[n] for n in names]

# def get_methods(ckpt):
#     M = {'Raw': lambda x: x, 'ICA': lambda x: B.ica(x, None, FS)}
#     try: import pywt; M['Wavelet'] = lambda x: B.wavelet(x, None, FS)
#     except ImportError: print('PyWavelets missing -> Wavelet skipped')
#     if ckpt:
#         import torch
#         from eegnipi.model import NIPDNet
#         dev = 'cuda' if torch.cuda.is_available() else 'cpu'
#         net = NIPDNet(); net.load_state_dict(torch.load(ckpt, map_location='cpu')); net.to(dev).eval()
#         def prop(x):
#             out = []
#             with torch.no_grad():
#                 for i in range(0, len(x), 64): out.append(net(torch.tensor(x[i:i + 64], dtype=torch.float32, device=dev))[0].cpu().numpy())
#             return np.concatenate(out)
#         M['Proposed'] = prop
#     return M

# def decode(F, y, groups):
#     clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000))
#     return float(cross_val_score(clf, F, y, cv=GroupKFold(min(5, len(np.unique(groups)))), groups=groups).mean())

# def eoec_report(X, y, subj, clean, methods):
#     occ, fr, tmp, ctr = chs(['O1', 'O2', 'Pz']), chs(['Fp1', 'Fp2']), chs(['T3', 'T4', 'F7', 'F8']), chs(['C3', 'Cz', 'C4'])
#     alpha = lambda Xm: np.log10(bandpow(Xm, 8, 13)[:, occ].mean(1))
#     react = lambda a, idx: a[idx & (y == 1)].mean() - a[idx & (y == 0)].mean()
#     raw = X; ref = react(alpha(raw), clean)
#     print(f'EO/EC: {len(y)} epochs, {(~clean).sum()} contaminated, {len(np.unique(subj))} subjects; reference alpha reactivity (clean epochs, raw) = {ref:.3f} log10 units')
#     rows = []
#     for name, fn in methods.items():
#         Xm = raw if name == 'Raw' else fn(raw); a = alpha(Xm); c = ~clean
#         r = dict(method=name, alpha_react=react(a, np.ones(len(y), bool)), alpha_react_err=abs(react(a, np.ones(len(y), bool)) - ref),
#                  acc_all=decode(_features(Xm, FS), y, subj), acc_contam=decode(_features(Xm[c], FS), y[c], subj[c]))
#         def ratio(chn, lo, hi): return float(np.median(bandpow(Xm[c], lo, hi)[:, chn].mean(1) / bandpow(raw[c], lo, hi)[:, chn].mean(1)))
#         r.update(frontal_slow_ratio=ratio(fr, 1, 4), temporal_hf_ratio=ratio(tmp, 30, 45), occ_alpha_ratio=ratio(occ, 8, 13), central_mu_ratio=ratio(ctr, 8, 13))
#         rows.append(r)
#     return pd.DataFrame(rows), ref

# def mi_report(X, y, subj, clean, methods):
#     c3, c4 = IDX['C3'], IDX['C4']
#     def li(Xm):
#         mu = np.log10(bandpow(Xm, 8, 13)); l = mu[:, c4] - mu[:, c3]
#         for s in np.unique(subj): l[subj == s] -= l[subj == s].mean()            # remove each subject's own baseline asymmetry
#         return l
#     def d(l, yy): a, b = l[yy == 0], l[yy == 1]; return float((a.mean() - b.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2))
#     rows = []
#     for name, fn in methods.items():
#         Xm = X if name == 'Raw' else fn(X); l = li(Xm)
#         rows.append(dict(method=name, d_all=d(l, y), d_contam=d(l[~clean], y[~clean]), d_clean=d(l[clean], y[clean])))
#     return pd.DataFrame(rows)

# if __name__ == '__main__':
#     p = argparse.ArgumentParser(); p.add_argument('--eoec'); p.add_argument('--mi'); p.add_argument('--ckpt'); p.add_argument('--out', default='outputs/physio'); a = p.parse_args()
#     methods = get_methods(a.ckpt); pd.set_option('display.width', 200)
#     if a.eoec:
#         d = np.load(a.eoec); X, y, subj = d['X'].astype(np.float32), d['y'], d['subj'];  clean = d['clean'].astype(bool) if 'clean' in d else keep_mask(X)
#         df, ref = eoec_report(X, y, subj, clean, methods); df.to_csv(a.out + '_eoec.csv', index=False)
#         print('\nEO vs EC. alpha_react_err = distance from the clean-epoch reactivity (lower=better); acc_* = subject-wise decoding;'
#               '\nratios on contaminated epochs: frontal_slow and temporal_hf should fall, occ_alpha and central_mu should stay near 1.')
#         print(df.round(3).to_string(index=False))
#     if a.mi:
#         d = np.load(a.mi); X, y, subj = d['X'].astype(np.float32), d['y'], d['subj']; clean = keep_mask(X)
#         df = mi_report(X, y, subj, clean, methods); df.to_csv(a.out + '_mi.csv', index=False)
#         print(f'\nMotor imagery: {len(y)} epochs, {(~clean).sum()} contaminated. Mu lateralisation effect size (Cohen d, left vs right):')
#         print(df.round(3).to_string(index=False))


In [42]:
# import os, shutil
# src = None
# for root, dirs, files in os.walk('/kaggle/input'):
#     if 'train.py' in files: src = root; break
# print('code source:', src)
# shutil.rmtree('/kaggle/working/proj', ignore_errors=True); shutil.copytree(src, '/kaggle/working/proj')
# os.chdir('/kaggle/working/proj'); print(os.getcwd())
# for f, t in [('train.py', 'lam_scale'), ('data_real.py', 'norm(w)'), ('eegnipi/nipi.py', 'bincount(labels).min() < 5'),
#              ('physio_validation.py', 'def eoec_report'), ('build_data.py', 'def collect')]:
#     print(f, 'OK' if os.path.exists(f) and t in open(f).read() else 'OUTDATED/MISSING -> redo Step 0')

In [43]:
# !pip -q install PyWavelets mne
# !python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

In [44]:
# !python build_data.py

In [45]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --lam-cov 2.0 --lam-scale 4 --drop erp --tag _lamx4

In [46]:
# !python physio_validation.py --eoec data/eoec_all.npz --mi data/mi_all.npz --ckpt outputs/best_nipdnet_lamx4.pt --out outputs/physio

In [47]:
# import numpy as np, sys
# sys.path.insert(0, '.')
# from physio_validation import *
# d = np.load('data/eoec_all.npz'); X = d['X'].astype(np.float32); clean = d['clean'].astype(bool)
# Y = get_methods('outputs/best_nipdnet_lamx4.pt')['Proposed'](X)
# fr = chs(['Fp1', 'Fp2'])
# sl, sy = bandpow(X, 1, 4)[:, fr].mean(1), bandpow(Y, 1, 4)[:, fr].mean(1)
# for lo, hi, nm in [(0, .5, 'lowest 50% frontal slow power'), (.5, .9, 'middle 40%'), (.9, 1, 'top 10% (blink-heavy)')]:
#     m = (sl >= np.quantile(sl, lo)) & (sl <= np.quantile(sl, hi))
#     print(f'{nm}: median frontal 1-4 Hz ratio = {np.median(sy[m] / sl[m]):.3f}')
# N = len(X); rel = np.linalg.norm((Y - X).reshape(N, -1), axis=1) / np.linalg.norm(X.reshape(N, -1), axis=1)
# print('relative change |Y-X|/|X|: clean epochs', round(float(np.median(rel[clean])), 3), '| contaminated', round(float(np.median(rel[~clean])), 3))
# print('median frontal peak-to-peak (uV): raw', round(float(np.median(np.ptp(X[:, fr], -1))), 1), '-> proposed', round(float(np.median(np.ptp(Y[:, fr], -1))), 1))

In [48]:
# import numpy as np, sys
# sys.path.insert(0, '.')
# from physio_validation import *
# from eegnipi.artifacts import contaminate
# d = np.load('data/eoec_all.npz'); X = d['X'].astype(np.float32); cl = d['clean'].astype(bool)
# fr = chs(['Fp1', 'Fp2'])
# f = lambda A: np.median(bandpow(A, 1, 4)[:, fr].mean(1))
# print('REAL: frontal 1-4 Hz power, contaminated / clean epochs =', round(f(X[~cl]) / f(X[cl]), 1))
# C = X[cl]; net = get_methods('outputs/best_nipdnet_lamx4.pt')['Proposed']
# for snr in [5, 0, -5, -10, -15]:
#     noisy, art, _ = contaminate(C, 'eog', snr, FS, 1); out = net(noisy)
#     print(f'SIM EOG at {snr:>3} dB: noisy/clean = {f(noisy) / f(C):7.1f}   model output/clean = {f(out) / f(C):7.1f}')

In [49]:
# import numpy as np, sys
# from scipy.stats import spearmanr
# sys.path.insert(0, '.')
# from physio_validation import *
# from eegnipi.artifacts import contaminate
# d = np.load('data/eoec_all.npz'); X = d['X'].astype(np.float32); cl = d['clean'].astype(bool)
# C = X[cl]; noisy, _, _ = contaminate(C, 'eog', 0, FS, 1)
# real = np.median(bandpow(X[~cl], 1, 4), 0) / np.median(bandpow(C, 1, 4), 0)
# sim = np.median(bandpow(noisy, 1, 4), 0) / np.median(bandpow(C, 1, 4), 0)
# print('channel   real   sim')
# for n, i in IDX.items(): print(f'{n:7s} {real[i]:6.1f} {sim[i]:6.1f}')
# print('Spearman(real, sim) across channels =', round(float(spearmanr(real, sim)[0]), 2))

In [50]:
# import numpy as np, sys
# sys.path.insert(0, '.')
# from physio_validation import *
# d = np.load('data/eoec_all.npz'); X = d['X'].astype(np.float32)
# fr, oc = chs(['Fp1', 'Fp2']), chs(['O1', 'O2'])
# sl = bandpow(X, 1, 4); fo = sl[:, fr].mean(1) / sl[:, oc].mean(1)
# m = fo >= np.quantile(fo, 0.9); Xb = X[m]
# print(f'blink-dominant epochs: {m.sum()}, median frontal ptp (uV) = {np.median(np.ptp(Xb[:, fr], -1)):.0f}, median Fp/O slow ratio = {np.median(fo[m]):.1f}')
# for name, fn in get_methods('outputs/best_nipdnet_lamx4.pt').items():
#     Y = Xb if name == 'Raw' else fn(Xb)
#     r = lambda ch, lo, hi: np.median(bandpow(Y, lo, hi)[:, ch].mean(1) / bandpow(Xb, lo, hi)[:, ch].mean(1))
#     print(f'{name:9s} frontal 1-4 Hz ratio = {r(fr, 1, 4):.3f}   occipital alpha ratio = {r(oc, 8, 13):.3f}')

In [51]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 80 --no-line --lam-cov 2.0 --lam-scale 4 --drop erp --snr-min -20 --tag _wide

In [52]:
# import numpy as np, sys
# sys.path.insert(0, '.')
# from physio_validation import *
# d = np.load('data/eoec_all.npz'); X = d['X'].astype(np.float32)
# fr, oc = chs(['Fp1', 'Fp2']), chs(['O1', 'O2'])
# sl = bandpow(X, 1, 4); fo = sl[:, fr].mean(1) / sl[:, oc].mean(1)
# m = fo >= np.quantile(fo, 0.9); Xb = X[m]
# print(f'blink-dominant epochs: {m.sum()}')
# for name, fn in get_methods('outputs/best_nipdnet_wide.pt').items():
#     Y = Xb if name == 'Raw' else fn(Xb)
#     r = lambda ch, lo, hi: np.median(bandpow(Y, lo, hi)[:, ch].mean(1) / bandpow(Xb, lo, hi)[:, ch].mean(1))
#     print(f'{name:9s} frontal 1-4 Hz ratio = {r(fr, 1, 4):.3f}   occipital alpha ratio = {r(oc, 8, 13):.3f}')

In [53]:
# !python bootstrap_ci.py --data data/eoec_clean.npz --ckpt outputs/best_nipdnet_lamx4.pt --snr 0 --B 100 --no-line

In [54]:
# import os, subprocess
# os.chdir('/kaggle/working/proj')
# need = ['nipi_validation.py', 'outputs/best_nipdnet_lamx4.pt', 'data/eoec_clean.npz']
# ok = {f: os.path.exists(f) for f in need}; print(ok)
# if all(ok.values()):
#     subprocess.run('mkdir -p /kaggle/working/keep && cp outputs/*.pt outputs/*.csv /kaggle/working/keep/', shell=True)
#     r = subprocess.run('python nipi_validation.py --data data/eoec_clean.npz --ckpt outputs/best_nipdnet_lamx4.pt --no-line', shell=True, capture_output=True, text=True)
#     print(r.stdout[-5000:]); print(r.stderr[-1500:])

In [55]:
# import pandas as pd, numpy as np
# from scipy.stats import spearmanr
# df = pd.read_csv('outputs/nipi_validation.csv')
# print(df.pivot_table(index=['kind', 'snr'], columns='method', values='Joint5').round(3).to_string())
# print(df.loc[df.groupby(['kind', 'snr']).Joint5.idxmax(), ['kind', 'snr', 'method', 'Joint5']].to_string(index=False))
# d2 = df[df.method != 'Lowpass_8Hz']
# print('Spearman NIPI5 vs acc_den without Lowpass_8Hz:', round(float(spearmanr(d2.NIPI5, d2.acc_den)[0]), 2))
# print(df.groupby('method').Joint5.mean().round(3).sort_values(ascending=False))

In [56]:
# import glob, os, numpy as np
# from scipy.io import loadmat
# files = sorted(glob.glob('/kaggle/input/**/Biosource/*', recursive=True)); print(files)
# for f in files:
#     if f.endswith('.mat'):
#         try:
#             m = loadmat(f); print('\n', os.path.basename(f))
#             for k, v in m.items():
#                 if not k.startswith('__'): print('  ', k, getattr(v, 'shape', None), getattr(v, 'dtype', None), (round(float(np.min(v)), 2), round(float(np.max(v)), 2)) if isinstance(v, np.ndarray) and v.dtype.kind in 'fi' and v.size else '')
#         except Exception as e: print('\n', os.path.basename(f), 'ERROR', str(e)[:150])
# for f in files:
#     if f.endswith('.m') and any(s in f for s in ('elpos19chch', 'meas19chch')):
#         print('\n=====', os.path.basename(f)); print(open(f, errors='ignore').read()[:1200])

In [57]:
# import glob, re, numpy as np
# from scipy.io import loadmat
# d = '/kaggle/input/datasets/adityabhoy/biosourceeegdataset/Biosource/'
# for f in sorted(glob.glob(d + '*.m')):
#     t = open(f, errors='ignore').read()
#     print(f.split('/')[-1], '->', [l.strip() for l in t.splitlines() if re.search(r'(?i)\bfs\b|freq|sampl|\bhz\b|rate', l)][:8])
# print('\n===== meas21avr.m'); print(open(d + 'meas21avr.m', errors='ignore').read()[:900])
# S = loadmat(d + 'sig_ictal.mat')['sig_ictal']; print('\nictal per-channel std:', np.round(S.std(1), 1))
# for n in ('spike1', 'spike2'):
#     A = loadmat(d + n + '.mat')['sig_av']; i = np.unravel_index(np.abs(A).argmax(), A.shape); print(n, 'max |amp| at channel idx', i[0], 'sample', i[1])

In [58]:
# !python build_biosource.py --dir /kaggle/input/datasets/adityabhoy/biosourceeegdataset/Biosource --out data/biosource.npz

In [59]:
# import numpy as np
# from scipy.signal import welch
# import matplotlib.pyplot as plt
# from eegnipi.montage import CH, IDX
# d = np.load('data/biosource.npz'); X = d['X_spike']; pk = d['spike_peak']
# for i in range(2):
#     x = X[i]; f, P = welch(x, 128, nperseg=256, axis=-1); tot = P.sum(-1)
#     hf = P[:, f > 25].sum(-1) / tot; sl = P[:, (f >= 1) & (f < 4)].sum(-1)
#     top = [CH[j] for j in np.argsort(-np.abs(x).max(1))[:3]]
#     print(f'spike{i+1}: peak {np.abs(x).max():.0f} uV | median ptp {np.median(np.ptp(x, -1)):.0f} | HF(>25 Hz) share {np.median(hf / 1):.3f} | Fp/O slow-power ratio {sl[[IDX["Fp1"], IDX["Fp2"]]].mean() / sl[[IDX["O1"], IDX["O2"]]].mean():.1f} | top-3 channels {top}')
# fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True); t = np.arange(512) / 128
# for i in range(2):
#     for c in range(19): ax[i].plot(t, X[i, c], lw=.7)
#     ax[i].axvline(pk[i] / 128, color='k', ls=':'); ax[i].set_title(f'spike{i+1}, all 19 channels')
# fig.tight_layout(); fig.savefig('outputs/biosource_spikes_overview.png', dpi=130)

In [60]:
# import numpy as np
# from scipy.signal import welch
# import matplotlib.pyplot as plt
# from eegnipi.montage import CH, IDX
# d = np.load('data/biosource.npz'); X = d['X_ictal']; t = np.arange(512) / 128
# for i, x in enumerate(X):
#     f, P = welch(x, 128, nperseg=256, axis=-1); tot = P.sum(-1); sl = P[:, (f >= 1) & (f < 4)].sum(-1)
#     hf = np.median(P[:, f > 25].sum(-1) / tot); pf = f[P.mean(0)[(f >= 1)].argmax() + 1] if False else f[1:][P.mean(0)[1:].argmax()]
#     print(f'epoch{i}: median ptp {np.median(np.ptp(x, -1)):.0f} uV | HF(>25 Hz) share {hf:.3f} | Fp/O slow-power ratio {sl[[IDX["Fp1"], IDX["Fp2"]]].mean() / sl[[IDX["O1"], IDX["O2"]]].mean():.1f} | dominant freq {pf:.1f} Hz | strongest channels {[CH[j] for j in np.argsort(-np.ptp(x, -1))[:3]]}')
# fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
# for i in range(3):
#     for c in range(19): ax[i].plot(t, X[i, c] + 0 * c, lw=.7)
#     ax[i].set_title(f'ictal epoch {i} (all 19 channels, overlaid)')
# fig.tight_layout(); fig.savefig('outputs/biosource_ictal_overview.png', dpi=130)

In [61]:
# # 1. Find the file in the input directory and copy it to the working project folder
# !find /kaggle/input -name biosource_case.py -exec cp {} /kaggle/working/proj/ \;

# # 2. Verify it copied successfully
# !ls /kaggle/working/proj/biosource_case.py

In [62]:
# import glob, os, shutil
# os.chdir('/kaggle/working/proj'); os.makedirs('outputs', exist_ok=True)
# src = glob.glob('/kaggle/input/**/best_nipdnet_lamx4.pt', recursive=True); print('found:', src)
# if src: shutil.copy(src[0], 'outputs/best_nipdnet_lamx4.pt'); print('copied ->', os.path.getsize('outputs/best_nipdnet_lamx4.pt'), 'bytes')
# print('biosource.npz exists:', os.path.exists('data/biosource.npz'))

In [63]:
# !python biosource_case.py --data data/biosource.npz --ckpt outputs/best_nipdnet_lamx4.pt --out outputs/biosource

In [64]:
# import numpy as np, torch
# from scipy.signal import welch
# from eegnipi.model import NIPDNet
# from eegnipi.montage import IDX
# d = np.load('data/biosource.npz'); XB = np.concatenate([d['X_ictal'], d['X_spike'][1:2]]).astype(np.float32)
# net = NIPDNet(); net.load_state_dict(torch.load('outputs/best_nipdnet_lamx4.pt', map_location='cpu')); net.eval()
# fp = [IDX['Fp1'], IDX['Fp2']]; m = lambda f: (f >= 1) & (f < 4)
# def ratio(Y, X):
#     f, P = welch(Y, 128, nperseg=256, axis=-1); f, Q = welch(X, 128, nperseg=256, axis=-1)
#     return float(np.median(P[:, fp][..., m(f)].sum((-1, -2)) / Q[:, fp][..., m(f)].sum((-1, -2))))
# for sign in (1, -1):
#     X = (sign * XB).astype(np.float32)
#     with torch.no_grad(): Y = net(torch.tensor(X))[0].numpy()
#     print(f'polarity {sign:+d}: frontal 1-4 Hz ratio ictal = {ratio(Y[:3], X[:3]):.3f} | spike2 = {ratio(Y[3:], X[3:]):.3f} | max Fp amplitude after (ictal) = {np.abs(Y[:3][:, fp]).max():.0f} uV')

In [65]:
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 0 --no-line --tag _lamx4 --drop erp

In [66]:
# !pip -q install PyWavelets mne
# !python build_data.py
# !python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 0 --no-line --tag _lamx4 --drop erp

In [67]:
import os, shutil, glob
src = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.py' in files: src = root; break
print('code:', src)
shutil.rmtree('/kaggle/working/proj', ignore_errors=True); shutil.copytree(src, '/kaggle/working/proj')
os.chdir('/kaggle/working/proj'); os.makedirs('outputs', exist_ok=True)
ck = glob.glob('/kaggle/input/**/best_nipdnet_lamx4.pt', recursive=True); print('checkpoint:', ck)
shutil.copy(ck[0], 'outputs/best_nipdnet_lamx4.pt')
print([f for f in ['bootstrap_ci.py', 'nipi_validation.py', 'build_biosource.py', 'biosource_case.py'] if os.path.exists(f)])

code: /kaggle/input/datasets/adityabhoy/eeg-nipi-code1/eeg_nipi
checkpoint: ['/kaggle/input/datasets/adityabhoy/eeg-nipi-outputs/keep/best_nipdnet_lamx4.pt']
['bootstrap_ci.py', 'nipi_validation.py', 'build_biosource.py', 'biosource_case.py']


In [68]:
!pip -q install PyWavelets mne
!python build_data.py
!python build_biosource.py --dir /kaggle/input/datasets/adityabhoy/biosourceeegdataset/Biosource --out data/biosource.npz
!python train.py --data data/train_mix.npz --test-data data/eoec_clean.npz --epochs 0 --no-line --tag _lamx4 --drop erp

1/4 Motor imagery, test subjects 1-20 (runs 04, 08, 12) ...
    MI test: kept 169 of 900 | classes [80 89] | subjects 9   (earlier run: 169 of 900)
2/4 Siena, first 2 recordings per patient, first 10 min each ...
    Siena: kept 886 of 2100 | patients 7   (earlier run: 886 of 2100)
3/4 Motor imagery, training subjects 21-60 -> training mix ...
    MI train: kept 326 of 1800 | training mix (1212, 19, 512)   (earlier run: 326 of 1800, mix 1212)
4/4 Eyes open / eyes closed, subjects 1-20 and 61-109 ...
    EO/EC: all 2070, clean 244, balanced test set 148 (classes [74 74])   (earlier run: 148 balanced)
DONE. Files in data/: ['eoec_all.npz', 'eoec_clean.npz', 'mi_all.npz', 'mi_clean.npz', 'siena_all.npz', 'siena_clean.npz', 'train_mix.npz']
spikes (2, 19, 512) peak sample (of 512): [256, 248] | ictal epochs (3, 19, 512)
spike ptp (uV, median over channels): [139.6, 206.6] | ictal ptp: [88.8, 95.9, 98.3]
train 1030 val 182 test 148
                    ARR     CC   NIPI  Joint
method        

In [69]:
!python run_final.py

[OK] train             11.6 min
[OK] bootstrap         24.5 min
[OK] nipi_validation    2.8 min
[OK] physio             1.1 min
[OK] blink              0.1 min
[OK] biosource          1.7 min

ZIPPED -> final_outputs.zip. Download it from the Output panel, then upload it to Claude.

--- main table (train log tail) ---
ep79 loss=2.653 ARR=0.75 NIPI=0.608 Joint=0.672 spectral=0.69 phase=0.92 connect=0.82 geometry=0.23 erp=0.87 entropy=0.49 task=nan
                    ARR     CC   NIPI  Joint
method                                      
Proposed_NIPDNet  0.580  0.823  0.721  0.625
NLMS              0.442  0.821  0.749  0.461
Wavelet           0.308  0.739  0.685  0.327
ICA               0.353  0.685  0.421  0.251
Notch_50          0.227  0.760  0.708  0.247
Lowpass_8Hz       0.248  0.660  0.114  0.077
Noisy_input       0.000  0.696  0.677  0.000



In [70]:
!pip -q install EMD-signal

In [71]:
%%writefile train_ae.py
import numpy as np, torch, torch.nn as nn, os
from torch.utils.data import DataLoader, TensorDataset
from eegnipi.artifacts import contaminate

class EEGAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(19, 32, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 16, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
            nn.Upsample(scale_factor=2), nn.Conv1d(16, 32, 7, padding=3), nn.ReLU(),
            nn.Upsample(scale_factor=2), nn.Conv1d(32, 19, 7, padding=3)
        )
    def forward(self, x): return self.net(x)

print("Loading training data for Denoising Autoencoder...")
d = np.load('data/train_mix.npz'); X_clean = d['X'].astype(np.float32)
noisy, _, _ = contaminate(X_clean, 'mixed', 0, 128, seed=42)
dataset = TensorDataset(torch.tensor(noisy), torch.tensor(X_clean))
loader = DataLoader(dataset, batch_size=64, shuffle=True)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
ae = EEGAutoencoder().to(dev)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)

print("Training Autoencoder (20 Epochs)...")
for ep in range(20):
    ae.train(); loss_sum = 0
    for x, y in loader:
        opt.zero_grad()
        loss = nn.MSELoss()(ae(x.to(dev)), y.to(dev))
        loss.backward(); opt.step()
        loss_sum += loss.item()
    if ep % 5 == 0: print(f"Ep {ep} Loss: {loss_sum/len(loader):.3f}")

os.makedirs('outputs', exist_ok=True)
torch.save(ae.state_dict(), 'outputs/ae_baseline.pt')
print("Saved outputs/ae_baseline.pt")

Writing train_ae.py


In [72]:
!python train_ae.py

Loading training data for Denoising Autoencoder...
Training Autoencoder (20 Epochs)...
Ep 0 Loss: 105.449
Ep 5 Loss: 38.783
Ep 10 Loss: 32.711
Ep 15 Loss: 30.331
Saved outputs/ae_baseline.pt


In [73]:
%%writefile run_extra_baselines.py
import numpy as np, pandas as pd, torch, torch.nn as nn, os, shutil
from PyEMD import EMD
from multiprocessing import Pool
from eegnipi.artifacts import contaminate
from eegnipi.nipi import nipi, artifact_suppression, joint_score, COMPONENTS

# 1. Setup Autoencoder
class EEGAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(19, 32, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 16, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
            nn.Upsample(scale_factor=2), nn.Conv1d(16, 32, 7, padding=3), nn.ReLU(),
            nn.Upsample(scale_factor=2), nn.Conv1d(32, 19, 7, padding=3)
        )
    def forward(self, x): return self.net(x)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
ae = EEGAutoencoder().to(dev)
if os.path.exists('outputs/ae_baseline.pt'):
    ae.load_state_dict(torch.load('outputs/ae_baseline.pt', map_location=dev))
ae.eval()

def run_ae(noisy, refs=None, fs=128):
    with torch.no_grad():
        return ae(torch.tensor(noisy, dtype=torch.float32).to(dev)).cpu().numpy()

# 2. Setup EMD (Multi-processed)
def _emd_channel(sig):
    imfs = EMD().emd(sig, max_imf=3)
    return np.sum(imfs[1:-1], axis=0) if imfs.shape[0] > 2 else sig

def run_emd(noisy, refs=None, fs=128):
    flat_sigs = noisy.reshape(-1, noisy.shape[-1])
    with Pool() as p:
        out_flat = p.map(_emd_channel, flat_sigs)
    return np.array(out_flat).reshape(noisy.shape)

extra_methods = {'Denoising_Autoencoder': run_ae, 'EMD': run_emd}

# 3. Evaluate Semi-Synthetic (Mixed 0dB for Table 1 and 3)
print("Evaluating Extra Baselines on Semi-Synthetic (Takes ~5 mins due to EMD)...")
d = np.load('data/eoec_clean.npz')
X_test = d['X'].astype(np.float32); y_test = d['y']
old_csv = pd.read_csv('outputs/final_comparison_final.csv')
new_rows = []

for name, fn in extra_methods.items():
    print(f"Running {name} on mixed 0dB...")
    noisy, art, refs = contaminate(X_test, 'mixed', 0, 128, seed=1)
    Y = fn(noisy, refs, 128)
    n = nipi(X_test, Y, y_test, 128, parts=COMPONENTS)
    
    # FIXED: Extract values from the dictionary correctly
    art_metrics = artifact_suppression(X_test, noisy, Y)
    arr = float(art_metrics.get('ARR', 0.0))
    cc = float(art_metrics.get('CC', 0.0))
    rmse = float(art_metrics.get('RMSE', 0.0))
    snr_gain = float(art_metrics.get('SNR_gain', 0.0))
    
    j = joint_score(arr, n['NIPI'])
    row = {'method': name, 'kind': 'mixed', 'snr': 0, 'ARR': arr, 'CC': cc, 'RMSE': rmse, 'SNR_gain': snr_gain, 'NIPI': n['NIPI'], 'Joint': j}
    for c in COMPONENTS: row[c] = n.get(c, float('nan'))
    new_rows.append(row)

df_new = pd.DataFrame(new_rows)
# Align columns 
for col in old_csv.columns:
    if col not in df_new.columns: df_new[col] = np.nan

df_combined = pd.concat([old_csv, df_new], ignore_index=True)
df_combined.to_csv('outputs/final_comparison_complete.csv', index=False)
print("\n--- New Table 1/3 Additions ---")
print(df_new[['method', 'ARR', 'NIPI', 'Joint', 'spectral', 'phase']].round(3).to_string(index=False))

# 4. Evaluate BioSource Spike (Mixed 0dB for Table 5)
print("\nEvaluating Extra Baselines on BioSource...")
d_bio = np.load('data/biosource.npz'); X_spike = d_bio['X_spike'][0:1].astype(np.float32)
bio_rows = []
noisy, art, refs = contaminate(X_spike, 'mixed', 0, 128, seed=1)
for name, fn in extra_methods.items():
    Y = fn(noisy, refs, 128)
    amp_ratio = np.abs(Y).max() / np.abs(X_spike).max()
    
    art_metrics = artifact_suppression(X_spike, noisy, Y)
    arr = float(art_metrics.get('ARR', 0.0))
    bio_rows.append({'method': name, 'snr': 0, 'amp_ratio': amp_ratio, 'ARR': arr})

print("\n--- New Table 5 Additions ---")
print(pd.DataFrame(bio_rows).round(3).to_string(index=False))

shutil.make_archive('final_outputs_complete', 'zip', 'outputs')
print("\nDONE. Download final_outputs_complete.zip")

Writing run_extra_baselines.py


In [74]:
!python run_extra_baselines.py

Evaluating Extra Baselines on Semi-Synthetic (Takes ~5 mins due to EMD)...
Running Denoising_Autoencoder on mixed 0dB...
Running EMD on mixed 0dB...

--- New Table 1/3 Additions ---
               method   ARR  NIPI  Joint  spectral  phase
Denoising_Autoencoder 0.665 0.376  0.481     0.269  0.693
                  EMD 0.338 0.484  0.398     0.297  0.783

Evaluating Extra Baselines on BioSource...

--- New Table 5 Additions ---
               method  snr  amp_ratio   ARR
Denoising_Autoencoder    0      0.473 0.733
                  EMD    0      0.704 0.532

DONE. Download final_outputs_complete.zip
